# SSMS vs Databricks Comparison Notes

**Purpose:** Track all SSMS export comparisons with Databricks tables to understand differences, root causes, and impact on the final `mifid2_report` output.

---

## Comparison Scorecard (as of 2026-06-10)

| # | Table | SSMS Rows | DBX Rows | Match % | Verdict | Impact on mifid2_report |
|---|-------|-----------|----------|---------|---------|-------------------------|
| 1 | `reg_ext_historysplitratio` | 1,048 | 1,048 (rebuilt) | 94.3% exact, 100% semantic | ✅ PASS | None — DBX now filtered to `IsCompletedOpenPositions=1` matching SSMS. Trade population already filtered on this. |
| 2 | `reg_ext_trade_getinstrument` | 16,243 | 16,243 | 99.9% (excl. Passport binary) | ✅ PASS | None — 13 NULL-vs-empty diffs in Industry (Forex/Commodity, no ISIN impact). |
| 3 | `reg_ext_trade_instrumentmetadata` | 15,510 | 16,243 | 93.8% on shared IDs | ✅ PASS (superset) | **POSITIVE**: 734 extra instruments in DBX → 431 have ISINs → 331,943 positions gain ISIN coverage that was NULL in SSMS. |
| 4 | `reg_instruments_ext` | 16,243 | 16,243 | 99.0% (8/12 cols exact) | ✅ PASS | None — key regulatory columns (InstrumentTypeID, BuyCurrencyID, SellCurrencyID, Tradable, ExchangeID, ContractExpire) ALL exact match. MiFID flags (IsMifidByESMA/FCA) only in DBX — 317 instruments in-scope. |
| 5 | `reg_ext_dictionarycurrency` | 15,487 | 16,271 | 99.8% on shared IDs (28 diffs) | ✅ PASS (superset) | None — 28 diffs are Name/Abbreviation updates (data freshness). CurrencyTypeID exact match. 784 extra currencies in DBX (new instruments). Report uses Abbreviation for PriceCurrency/QuantityCurrency — all shared IDs match. |
| 6 | `reg_ext_dictionarycurrencytype` | 10 | 10 | **100% exact** | ✅ PERFECT MATCH | None — all 10 asset class definitions identical. Maps InstrumentTypeID to Name (Forex/Stocks/etc.) for AssetClass field in report. |
| 7 | `reg_hedgeservertoliquidityaccount_ext` | 79 | 79 | **100% exact** | ✅ PERFECT MATCH | None — all hedge-server-to-account mappings identical. Used for Venue field derivation. |
| 8 | `reg_ext_liquidityproviders` | 190 | 190 | **100% exact** | ✅ PERFECT MATCH | None — all provider names/types identical. Used for LP identification in hedge report. |
| 9 | `reg_ext_liquidityaccountid` | 394 | 386 | 97.9% on shared IDs | ✅ PASS | 8 extra accounts in SSMS (IDs 151-153, 380-383, 100027) not in DBX source. NULL-vs-empty on IsActive. No active positions reference these 8. |
| 10 | `reg_liquidtyacount_ext` | 394 | 394 | 99.7% (1 diff) | ✅ PASS | 1 row: AccountID=210 reassigned from ProviderID 360→ 71 (data freshness). All other 393 rows exact match. |
| 11 | `mifid2_ext_mirror` | 2,807 | 2,807 (rebuilt) | **100% row count** | ✅ PASS (fixed) | SCD fan-out bug in backoffice_asof CTE produced 28,574 duplicates. Fixed: ROW_NUMBER dedup on CID. Same 2,807 distinct MirrorIDs, same 546 ParentCIDs, same date range. |
| 12 | `mifid2_ext_hedgeexecutionlog` | 251,870 | 251,870 | **100% on key cols** | ✅ PASS | Row count exact match. HedgeServerID, InstrumentID, IsBuy, LiquidityAccountID = 0 diffs. Units (8.9%) and ExecutionRate (3.6%) diffs = CSV precision truncation only. 56 Success diffs (0.02%). Report uses only HedgeServerID+LiquidityAccountID for Venue derivation. |
| 13 | `reg_regulationinoutdailydata` | 8,632,673 (cumulative) | 29,127 (stale) | N/A | ⚠️ UPSTREAM GAP | Gold pipeline `gold_regtech_reg_regulationinoutdailydata` stopped at 2023-05-15. SSMS has live daily appends (45 rows on 2026-06-09). DBX produces 0 rows for current dates. Low impact: only ~45 migration customers/day. **ACTION:** DE team must resume gold pipeline. See lineage below. |
| 14 | `reg_ext_dailymaxprices` | 16,909 (2026-09-10) | 16,017 (2026-06-09) | **100% instrument overlap** | ✅ PASS | Identical 16,017 InstrumentIDs in both. SSMS has 892 extra rows (2 price-date partitions for some instruments on 2026-09-10 vs 1 on 2026-06-09). Date-dependent behavior, not a logic error. Same 25 core columns. |
| 15 | `reg_ext_currencypricemaxdatewithsplit` | 16,909 (2026-09-10) | 16,017 (2026-06-09) | **100% instrument overlap** | ✅ PASS | Same pattern as #14. SSMS has 2 OccurredDateID partitions (20260609: 892, 20260610: 16,017). DBX has 1 partition (20260609: 16,017). 16 shared columns all present. |
| 16 | `mifid2_ext_customer` | 205,036 (2026-06-10) | 203,345 (2026-06-09) | **-0.8%** | ✅ PASS | Close match. Both 1:1 CID. Diff = 1,691 more customers on 2026-06-10 (more positions → more qualifying CIDs). Same logic confirmed. |
| 17 | `mifid2_ext_position` | 2,734,318 (2026-06-10) | 2,736,440 (2026-06-10) | **+0.08%** | ✅ PASS | Recomputed for same date: only +2,122 positions (+0.08%). Original +51.8% gap was entirely due to different trade dates (2026-06-09 was much busier). 127 extra CIDs from SCD dedup differences. |
| 18 | `mifid2_ext_positionchangelog` | 1,404,855 (2026-06-10) | ~1,404,855 (2026-06-10) | **~0%** | ✅ PASS (inferred) | Changelog is filtered by PositionIDs from position table. With position count matching at +0.08%, changelog will match proportionally. |
| 19 | `mifid2_report` (Close events) | 669 (EU, ReportID=1) | 669/669 joined | **100% TRN/Price/Venue** | ✅ PASS | All key fields validated. 8 sub-second TradingDateTime diffs. SellerCodeType 309/669 = masked IDType limitation. |
| 20 | `mifid2_report` (Open events) | 884 (EU, ReportID=1) | 884/884 joined | **100% TRN/Price/Venue** | ✅ PASS | All key fields validated. 7 sub-second TradingDateTime diffs. BuyerCodeType 660/884 = masked IDType limitation. |
| 21 | `mifid2_report` (UK Close) | 331 (UK, ReportID=2) | 331/331 joined | **100% TRN/Price** | ✅ PASS | Full join on UK report rows. |

---

## Detailed Findings

### 1. reg_ext_historysplitratio
- **Root cause of row diff:** SSIS filters `IsCompletedOpenPositions = 1` (1,048 rows). DBX originally loaded full table (16,630). Fixed: added filter to NB01 cell 4.
- **60 value diffs:** `AmountRatio` floating-point precision (SSMS rounds to 4dp, DBX stores full precision). All other columns match exactly.
- **Downstream impact:** `mifid2_report_trade_population` split_ratio CTE already filters `IsCompletedOpenPositions=1` → zero impact on final report.

### 2. reg_ext_trade_getinstrument
- **Passport column:** BINARY in DBX vs hex string in CSV — cannot compare, excluded. Not used in mifid2_report.
- **Industry (5,001 apparent mismatches):** SSMS exports SQL NULL as literal string "NULL" in CSV. After normalizing: only 13 true diffs (NULL vs empty-string for Forex/Commodity).
- **Downstream impact:** `mifid2_report` uses InstrumentID, BuyCurrencyID, SellCurrencyID, InstrumentTypeID from this table. ALL match exactly → zero report impact.

### 3. reg_ext_trade_instrumentmetadata
- **734 extra instruments in DBX:** All present in GetInstrument (both systems). SSMS InstrumentMetaData query is more restrictive.
- **453 of 734 have active positions** → 488,590 positions (11.8% of trade population).
- **431 of 453 have valid ISINs** → 331,943 positions gain ISIN coverage in DBX that was NULL in SSMS.
- **967 column-level diffs on shared IDs:** Cusip (CSV scientific notation truncation), Symbol/Exchange (source data updated between snapshots), ISIN/ISINCountry (NULL-vs-empty).
- **Downstream impact on mifid2_report:** Report uses ISINCode, ISINCountryCode, InstrumentTypeID from InstrumentMetaData for field population. DBX produces **better ISIN coverage** for 331,943 positions → improvement, not regression.

### 15. reg_ext_currencypricemaxdatewithsplit
- **Trade dates compared:** SSMS = 2026-09-10, DBX = 2026-06-09
- **Row count:** SSMS 16,909 | DBX 16,017 | diff = 892 (same as #14)
- **InstrumentID coverage:** IDENTICAL (16,017 shared, 0 exclusive)
- **OccurredDateID distribution:**
  - SSMS: 20260610 = 16,017 rows + 20260609 = 892 rows (2 active max-date partitions)
  - DBX: 20260609 = 16,017 rows (1 active max-date partition)
- **Column alignment:** 16 SSMS columns (excl. UpdateDate) all present in DBX. DBX adds `etr_y/ym/ymd` partition cols.
- **Relationship to #14:** Both tables derive from the same `currencypricemaxdate` source. This table has the summary columns (16); DailyMaxPrices has the full price-log columns (25+). Same 892 extra rows on same 892 instruments.
- **Downstream impact:** Provides the split-adjusted price reference for the report. Instrument coverage identical → zero impact.

### 14. reg_ext_dailymaxprices
- **Trade dates compared:** SSMS = 2026-09-10, DBX = 2026-06-09 (different dates, row counts expected to differ)
- **Row count:** SSMS 16,909 | DBX 16,017 | diff = 892
- **InstrumentID coverage:** IDENTICAL (16,017 shared, 0 in SSMS-only, 0 in DBX-only)
- **Granularity:** DBX = exactly 1 row/instrument. SSMS = 1.06 rows/instrument (892 instruments have 2 rows from 2 max-date partitions)
- **892 extra rows explained:** The source `currencypricemaxdate` table sometimes has 2 active date partitions per instrument. On 2026-09-10 some instruments have data from both partitions; on 2026-06-09 all have only 1. This is date-dependent, not a logic error.
- **Column alignment:** 25 SSMS columns all present in DBX. DBX has 8 extra columns (partition cols `etr_y/ym/ymd`, `SkewID`, `Volume`, `PriceRateInsertTime`, `USDConversionRateBid/AskSpreaded`, `USDConversionPriceRateID`) from the fuller bronze source.
- **Downstream impact on mifid2_report:** This table provides the price snapshot for open-price validation. InstrumentID coverage is identical → zero report impact.

### 13. reg_regulationinoutdailydata — UPSTREAM GAP
- **SSMS:** 8,632,673 rows (cumulative append since 2022-01-06, UpdateDate range 2022-01-06 → 2026-06-11)
- **DBX:** 29,127 rows (stale, ReportDate range 2023-01-03 → 2023-05-15, UpdateDate max 2023-05-16)
- **Gold source:** `main.regtech.gold_regtech_reg_regulationinoutdailydata` — same stale data (max 2023-05-15)
- **For report_date 2026-06-09:** SSMS has 45 rows, DBX has 0 rows
- **Root cause:** The Databricks gold pipeline that populates this table was stopped/decommissioned in May 2023. The SSMS/Synapse SSIS package continues to run daily.
- **Impact on mifid2_report:** Low. Only ~45 customers/day migrate between regulations. Feeds `reg_regulation_movments_positions` and `mifid2_ext_regchange_customer`/`mifid2_ext_regchange_position` (all 0 rows in DBX). These positions would report under their previous regulation.
- **Resolution:** DE team must resume the gold pipeline or create a new ingestion path from Synapse `RegReportDB.dbo.Reg_RegulationInOutDailyData`.

### 12. mifid2_ext_hedgeexecutionlog
- **Row count:** 251,870 = 251,870 (exact match)
- **EMSOrderID key match:** 251,868 of 251,870 rows join perfectly on EMSOrderID (99.99%)
- **Column-level analysis on joined rows:**
  - HedgeServerID: **0 diffs** (perfect)
  - InstrumentID: **0 diffs** (perfect)
  - IsBuy: **0 diffs** (perfect)
  - LiquidityAccountID: **0 diffs** (perfect)
  - Success: 56 diffs (0.02%)
  - ExecutionRate: 9,186 diffs (3.6%) — CSV truncation to 1-2dp vs DBX 8dp
  - Units: 22,512 diffs (8.9%) — CSV truncation to integer vs DBX 8dp decimal
- **CSV artifacts:** ProviderExecID rendered in scientific notation (2.60609E+13); ExecutionTime truncated to mm:ss.f format
- **Downstream impact on mifid2_report:** This table is used ONLY for Venue derivation via `HedgeServerID + LiquidityAccountID` JOIN. Both columns = perfect match. Units/Rate not consumed by the report. Zero impact.

### 11. mifid2_ext_mirror
- **Root cause of 10x inflation (28,574 vs 2,807):** `backoffice_asof` CTE joined to `bronze_etoro_history_backofficecustomer` SCD without deduplication. Overlapping SCD validity ranges per CID created fan-out (e.g., CID 12293650 had 48 overlapping SCD rows → each mirror row duplicated 48x).
- **Fix applied:** Added `ROW_NUMBER() OVER (PARTITION BY b.CID ORDER BY b.ValidFrom DESC) AS rn` + `WHERE rn = 1` to `backoffice_asof` CTE in NB05 cell 4.
- **After fix:** 2,807 rows = exact match with SSMS. Same 546 distinct ParentCIDs, same date range, 1:1 MirrorID.
- **Downstream impact on mifid2_report:** `mifid2_ext_mirror` is LEFT JOINed on MirrorID in the trade population to derive `IsCopy` and `TransmissionOfOrderIndicator`. The fan-out would have inflated the trade population JOIN. Now fixed → correct 1:1 match.
- **Same root cause as NB07:** `bronze_etoro_history_backofficecustomer` SCD has overlapping validity ranges — this is the third query fixed for this issue.

### 5. reg_ext_dictionarycurrency
- **Row count:** SSMS 15,487 (1 shifted CSV row) | DBX 16,271 | 784 extra in DBX
- **784 extra CurrencyIDs in DBX:** Same superset pattern as InstrumentMetaData — newer currencies added to source.
- **On 15,487 shared IDs:** Only 28 rows differ (0.2%)
  - Name: 26 diffs (renamed currencies, data freshness)
  - Abbreviation: 25 diffs (ticker changes)
  - EEAStockExchange: 1 diff
  - CurrencyTypeID: **0 diffs** (exact match)
- **Downstream impact on mifid2_report:** Report uses `Abbreviation` from DictionaryCurrency to populate `PriceCurrency`, `QuantityCurrency`, `NotionalCurrency1/2`, `StrikePriceCurrency`. The 25 abbreviation diffs are ticker updates on shared IDs — DBX has the current correct value. Zero logic errors.

### 4. reg_instruments_ext
- **Row count:** Exact match (16,243 = 16,243)
- **8/12 shared columns: PERFECT MATCH** — InstrumentTypeID, Tradable, InstrumentVisible, BuyCurrencyID, SellCurrencyID, ContractExpire, ExchangeID, VisibleInternallyOnly
- **4 columns with minor diffs (data freshness):** InstrumentDisplayName (120), ISINCode (43), Symbol (2), SymbolFull (1)
- **`IsFuture` column:** Present in SSMS only, not in DBX. Not the same as `ContractExpire` (both columns exist in SSMS, values differ). **CONFIRMED: NOT used in mifid2_report output** (no IsFuture column in final 100-col report; ExpiryDate/MaturityDate/AssetClass derived from InstrumentTypeID instead).
- **`IsMifidByESMA` / `IsMifidByFCA`:** Present in DBX only (from `gold_regtech_reg_instruments_scd`). Critical for MiFID scoping: 316 instruments = ESMA+FCA, 1 = FCA-only, 15,926 = not in scope.
- **Downstream impact on mifid2_report:** The report uses `IsMifidByESMA`/`IsMifidByFCA` to determine which positions are EU/UK-reportable. Since all regulatory-critical columns match perfectly, the MiFID scope determination is correct. `IsFuture` confirmed NOT used in report output fields — zero impact.

---

## Recurring Patterns (apply to all future comparisons)

| Pattern | Cause | Handling |
|---------|-------|----------|
| Literal "NULL" string in CSV | SSMS exports SQL NULL as text | Normalize with `NULLIF(col, 'NULL')` |
| Cusip scientific notation | SSMS/Excel numeric formatting | Known artifact, ignore |
| NULL vs empty string | Source storage difference | Treat as semantic match |
| Passport/binary columns | CSV cannot represent binary | Exclude from comparison |
| Row count: DBX > SSMS | DBX reads full bronze, SSIS has filters | Investigate if extra rows have positions |
| Floating-point precision | SSMS rounds, DBX stores full precision | Semantic match if within 1e-4 |

---

## Impact Assessment on mifid2_report Final Output

The `mifid2_report` table (100 columns) consumes:
- `mifid2_customer` → from `mifid2_ext_customer` (CID, RegulationID, Names, BirthDate, PIN_LEI)
- `mifid2_report_trade_population` → from `mifid2_ext_position` + `reg_ext_trade_getinstrument` + `reg_ext_trade_instrumentmetadata` + `reg_ext_historysplitratio` + `mifid2_ext_positionchangelog`

**Current assessment:** All compared tables show semantic parity or improvement vs SSMS. No logic errors detected. The final report will be **at least as correct as SSMS** and in some cases better (ISIN coverage).

### 19-21. mifid2_report (Final Report Validation)
- **Samples:** 1,000 Close events + 1,000 Open events from SSMS (fixed-width format, parsed in chat)
- **RegulationReportID semantics (CORRECTED):**
  - RegulationReportID = 1 → **EU Report** (eToro Europe/CySEC)
  - RegulationReportID = 2 → **UK Report** (eToro UK/FCA)
- **Flag logic (CORRECTED):**
  - `IsEUReport` = RegulationID IN (1, 2, 9) → feeds RegulationReportID=1
  - `IsUKReport` = RegulationID IN (2) → feeds RegulationReportID=2
  - RegulationID=11 (US/Apex) → NEITHER report
- **Bugs found and fixed:**
  1. **Price derivation (Close events):** Used `InitForexRate` (open price) instead of `EndForexRate` (close price). Fixed: `CASE WHEN OpenORClose='O' THEN InitForexRate ELSE EndForexRate END`
  2. **IsUKReport/IsEUReport flags inverted:** Original logic had RegID=1→BOTH, RegID=2→UK only, RegID=9→EU only. Correct: RegID=1→EU only, RegID=2→BOTH, RegID=9→EU only, RegID=11→NEITHER
  3. **Flag names swapped:** Column names were semantically reversed. Fixed: `IsEUReport` now correctly selects EU positions.
- **29 unmatched positions (resolved):** All were RegID=9 (CySEC) positions missing from EU report due to flag inversion. After fix: 669/669 Close events join.
- **Validation scores (EU Close, 669 rows):**
  - TRN: 669/669 (100%), Venue: 669/669 (100%), Price: 669/669 (100%)
  - PriceCurrency: 669/669 (100%), Quantity: 669/669 (100%)
  - TradingDateTime: 661/669 (98.8% — sub-second rounding)
  - BuyerCodeType: 669/669 (100%), SellerCodeType: 309/669 (46% — masked IDType limitation)
  - IsCopy: 669/669 (100%), ExecutingEntity: 669/669 (100%)
- **Known gaps (not logic errors):**
  - IDType/SellerCodeType/BuyerCodeType: All customers get IDType=3 (CONCAT) due to masked fallback. Resolves when PII access is granted.
  - TradingDateTime: 8-15 rows with sub-second rounding (harmless)
- **Final row counts (2026-06-10):**
  - EU Report (ReportID=1): 1,477,060 Close + 1,337,078 Open = 2,814,138
  - UK Report (ReportID=2): 356,838 Close + 307,050 Open = 663,888
  - Total: 3,478,026 rows

### TRN Suffix Patterns (fully decoded from SSMS)
| RegulationID | Report | OpenORClose | TRN Suffix |
|---|---|---|---|
| 1 (CySEC main) | EU (ID=1) | O | `{PosID}O` |
| 1 (CySEC main) | EU (ID=1) | C | `{PosID}C` |
| 2 (FCA) | EU (ID=1) | O | `{PosID}UKO` |
| 2 (FCA) | EU (ID=1) | C | `{PosID}UKC` |
| 9 (CySEC variant) | EU (ID=1) | O | `{PosID}OSC{DateID}` |
| 9 (CySEC variant) | EU (ID=1) | C | `{PosID}CSC{DateID}` |
| 2 (FCA) | UK (ID=2) | O | `{PosID}O` |
| 2 (FCA) | UK (ID=2) | C | `{PosID}C` |

---

---

## SP_MIFID2_Report Analysis (2026-06-11)

**Full stored procedure received and analyzed.** Key findings incorporated into NB08 cell `6eb54007`.

### Report Structure (5 INSERT Blocks in SP)

| # | INSERT Target | ReportID | OrigRegulationID | IsMifid Filter | Description |
|---|---|---|---|---|---|
| 1 | MIFID2_Report | 1 (EU) | 1 (CySEC) | `M.IsMifid = 1` | Standard EU report |
| 2 | MIFID2_Report | 2 (UK) | 2 (FCA) | `M.IsMifidByFCA = 1` | Standard UK report |
| 3 | MIFID2_Report | 1 (EU) | 2 (FCA) | `M.IsMifidByFCA = 1` AND `M.IsMifid = 1` | FCA→EU intercompany hedge (DSR-7188/8383) |
| 4 | MIFID2_Report | 1 (EU) | 9 (Seychelles) | `M.IsMifid = 1` | Seychelles→EU (DSR-6762) |
| 5 | MIFID2_ME_Report | 1 | 11 (ME) | `M.IsMifid = 1` | ME report (SEPARATE table!) |

### Key Field Differences by Report Flow

| Field | EU (RegID=1) | UK (RegID=2) | FCA→EU (RegID=2→EU) | Seychelles (RegID=9) |
|---|---|---|---|---|
| ExecutingEntity | `213800GIFQMSV7HROS23` | `213800FLAB1OVA8OHT72` | `213800GIFQMSV7HROS23` | `213800GIFQMSV7HROS23` |
| TradingCapacity | DEAL (AOTC if futures) | **AOTC always** | DEAL | DEAL |
| TransmissionOfOrder | false | **true** | false | false |
| CountryOfBranch | CY | **GB** | CY | CY |
| Buyer/Seller | Customer vs eToro EU | Customer vs eToro EU | **eToro UK vs eToro EU** (LEI only) | **eToro SC vs eToro EU** (LEI only) |
| DecisionMaker | Copy trade logic | Copy trade logic (UK LEI) | Empty | Empty |
| TRN suffix | `{PosID}{O/C}` | `{PosID}{O/C}` | `{PosID}UK{O/C}` | `{PosID}{O/C}SC{DateID}` |

### Exclusion Tables (Confirmed in DBX)

| SP Source | DBX Table | Data |
|---|---|---|
| `[ThirdParty_Fivetran].[Fivetran].[regulation].[regtech_excluded_instruments]` | `main.regtech_stg.silver_sharepoint_transactionreporting_regtech_excluded_instruments` | 1 instrument (ID 624) for `[MIFID2_Report]` |
| `[ThirdParty_Fivetran].[Fivetran].[regulation].[regtech_excluded_position_ids]` | `main.regtech_stg.silver_sharepoint_transactionreporting_regtech_excluded_position_ids` | **2,699 positions** for `[MIFID2_Report]` |
| `[ThirdParty_Fivetran].[Fivetran].[regtech].[regulation_report_excluded_cids]` | `main.regtech_stg.silver_sharepoint_transactionreporting_regulation_report_excluded_cids` | 47 CIDs (beta testers, UK report only) |
| `[ThirdParty_Fivetran].[google_sheets].[isin_for_instrumentid_341]` | `main.regtech_stg.silver_sharepoint_transactionreporting_isin_for_instrumentid_341` | 2 rows (InstrumentID 999991/999992 → Brent Oil futures ISINs) |
| `[dbo].[MIFID2_Instruments_To_Exclude]` | Not found separately — only InstrumentID 624 in excluded_instruments | Russian instruments for Real only |
| `[dbo].[FuturesMetaData]` | `main.trading.bronze_etoro_trade_futuresmetadata` | Confirmed (Multiplier, ExpirationDateTime, no CFICode column) |

### SP Corrections Applied to NB08 (2026-06-11)

1. ✅ ExecutingEntity: UK report now uses `213800FLAB1OVA8OHT72` (eToro UK)
2. ✅ TradingCapacity: UK = 'AOTC' always (was 'DEAL')
3. ✅ TransmissionOfOrderIndicator: EU='false', UK='true' (was NULL)
4. ✅ CountryOfBranch: EU='CY', UK='GB' (was NULL)
5. ✅ ReportStatus: 'NEWT' (was NULL)
6. ✅ ShortSellingIndicator: Conditional (Stocks/ETF AND Real AND sell side) (was always 'SELL')
7. ✅ PriceType: Indices(TypeID=4)='BSPS', else 'MNTR' (was always 'MONE')
8. ✅ FCA→EU intercompany: Buyer/Seller = eToro UK vs eToro EU LEIs
9. ✅ Seychelles: Buyer/Seller = eToro SC (`549300L7LPQNKJQ1IW32`) vs eToro EU LEIs
10. ✅ BuyerDecisionMaker/SellerDecisionMaker: populated for copy trades (was NULL)
11. ✅ InvestmentDecisionWithinFirm: 'ALG'/'ETOROBROKERAGE01' per SP (was NULL)
12. ✅ ExecutionWithinFirm: 'ALG'/'ETOROBROKERAGE01' (was NULL)
13. ✅ CommodityDerivativeIndicator: 'false' for commodities (was NULL)
14. ✅ BackReportingIndicator: 0 (was NULL)
15. ✅ TradingDateTime: seconds=0 bumped to 1 (FCA rejects :00)
16. ✅ PriceCurrency: CNH→CNY mapping added

### NOT Yet Applied (Awaiting Fixes/Decisions)

- ❌ IsMifid instrument filter: gold table `gold_regtech_reg_instruments_scd` only tags 317/5,378 instruments. SP confirms `IsMifid=1` IS correct filter. Awaiting DE fix.
- ❌ Exclusion filters: Tables confirmed available but not yet integrated into report SQL.
- ❌ IsFuture column: Not in DBX `reg_instruments_ext`. SP uses for TradingCapacity (futures→AOTC), Venue (XOFF), ExpiryDate, PriceMultiplier, CFICode.
- ❌ InstrumentClassification CFI codes: Complex branch-specific mapping deferred.
- ❌ Partial close logic: Complex OriginalPositionID/InitialUnits substitution deferred.
- ❌ FundType copy-trade granularity: People/Partners/Market → different InvestmentDecisionWithinFirm values.
- ❌ InstrumentMetaData_SpecialChar_Conversion: Special char handling (ä→a, commas→spaces).
- ❌ Reg_Instruments_Full_Description: IndexNameFullDescription for indices' UnderlyingIndexName.

### +0.8% Gap Hypothesis (Updated)

The remaining gap (DBX 3,014,698 vs SSMS 2,991,891 = +0.8%) likely composed of:
1. **2,699 excluded positions** (not yet filtered) → could account for ~5,398 report rows (O+C)
2. **1 excluded instrument** (ID 624) → minimal impact
3. **47 excluded CIDs** (UK only) → some rows removed from UK report
4. **Crypto/Forex instruments** (DBX has 205 crypto + 56 forex instruments) → may not be MiFID-reportable
5. **IsMifid filter** (when fixed) → will be the primary row-count reconciliation

---

*This cell is updated as each SSMS table comparison completes.*

# Module 8B: MIFID2_ext Customer Staging (Masked Fallback)

This notebook creates the MIFID2_ext customer staging tables using **masked customer tables** as a temporary fallback for structural testing.

**Replicates SSIS package:** `MIFID2.dtsx` Step 9 (truncate/reload)

**Fallback strategy (per repository `open_blockers_for_execution.md`):**
- Uses `main.general.bronze_etoro_customer_customer_masked` instead of `main.pii_data.bronze_etoro_customer_customer`
- Uses `main.general.bronze_etoro_history_customer_masked` instead of `main.pii_data.bronze_etoro_history_customer`
- Allowed for: schema profiling, row-count checks, join-path testing, structural validation
- Does NOT close identity-field parity gates (FirstName, LastName, BirthDate will contain masked values)

**When PII access is granted:** Change `use_masked_fallback = True` to `False` below. The queries will automatically switch to the real PII sources.

**Targets created:**
- `bi_output_regtechops_mifid2_ext_customer` — Customer snapshot (masked)
- `bi_output_regtechops_mifid2_ext_regchange_customer` — RegChange customer (masked)
- `bi_output_regtechops_mifid2_ext_position` — Position staging (depends on customer CID scope)
- `bi_output_regtechops_mifid2_ext_regchange_position` — RegChange position staging

In [0]:
dbutils.widgets.text("report_date", "2026-06-08", "Report Date (YYYY-MM-DD)")
dbutils.widgets.dropdown("use_masked_fallback", "true", ["true", "false"], "Use Masked Customer Fallback")

report_date = dbutils.widgets.get("report_date")
use_masked_fallback = dbutils.widgets.get("use_masked_fallback").lower() == "true"

# Source table configuration - swap when PII access is granted
if use_masked_fallback:
    CUSTOMER_TABLE = "main.general.bronze_etoro_customer_customer_masked"
    HISTORY_CUSTOMER_TABLE = "main.general.bronze_etoro_history_customer_masked"
    print("⚠️  MASKED FALLBACK MODE - structural testing only")
    print("    Identity fields (FirstName, LastName, BirthDate) contain masked values.")
    print("    This does NOT close final parity gates.")
else:
    CUSTOMER_TABLE = "main.pii_data.bronze_etoro_customer_customer"
    HISTORY_CUSTOMER_TABLE = "main.pii_data.bronze_etoro_history_customer"
    print("✓ PRODUCTION PII MODE - real identity data")

print(f"\nCustomer source: {CUSTOMER_TABLE}")
print(f"History source:  {HISTORY_CUSTOMER_TABLE}")
print(f"Report date:     {report_date}")

In [0]:
# MIFID2_ext_Customer: Customer snapshot for report-date
# SSIS parity: MIFID2.dtsx Step 9 - truncate/reload with:
#   - Customer/backoffice as-of logic: ValidFrom < EndDate AND ValidTo >= EndDate
#   - RegulationID IN (1,2,9,11), AccountTypeID NOT IN (7,9), LabelID NOT IN (26,30)
#   - CID scope constrained by report-day qualifying positions

customer_sql = f"""
CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_customer
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_ext_customer'
AS
WITH run_window AS (
  SELECT
    CAST('{report_date}' AS DATE) AS report_date,
    CAST('{report_date}' AS TIMESTAMP) AS start_ts,
    CAST(date_add(CAST('{report_date}' AS DATE), 1) AS TIMESTAMP) AS end_ts
),
-- CID scope from qualifying positions (trade + history)
position_cids AS (
  SELECT DISTINCT p.CID
  FROM main.bi_db.bronze_etoro_trade_positionforexternaluse p
  JOIN run_window w ON p.Occurred >= w.start_ts AND p.Occurred < w.end_ts
  UNION
  SELECT DISTINCT p.CID
  FROM main.trading.bronze_etoro_history_position_datafactory p
  JOIN run_window w
    ON ((p.CloseOccurred >= w.start_ts AND p.CloseOccurred < w.end_ts)
        OR (p.OpenOccurred >= w.start_ts AND p.OpenOccurred < w.end_ts AND p.CloseOccurred >= w.end_ts))
   AND p.OpenOccurred >= CAST('2015-04-26' AS TIMESTAMP)
),
customer_asof_ranked AS (
  SELECT
    c.CID,
    c.GCID,
    c.PlayerLevelID,
    c.PlayerStatusID,
    c.CountryID,
    h.LabelID,
    h.FirstName,
    h.LastName,
    h.BirthDate,
    b.RegulationID,
    b.AccountTypeID,
    b.Lei,
    c.CountryIDByIP,
    h.FirstName AS curFirstName,
    h.LastName AS curLastName,
    h.BirthDate AS curBirthDate,
    c.CitizenshipCountryID,
    ROW_NUMBER() OVER (PARTITION BY c.CID ORDER BY b.ValidFrom DESC, h.ValidFrom DESC) AS rn
  FROM {CUSTOMER_TABLE} c
  JOIN position_cids pc ON c.CID = pc.CID
  JOIN run_window w ON 1 = 1
  JOIN {HISTORY_CUSTOMER_TABLE} h
    ON c.CID = h.CID
   AND h.ValidFrom < w.end_ts
   AND h.ValidTo >= w.end_ts
  JOIN main.general.bronze_etoro_history_backofficecustomer b
    ON c.CID = b.CID
   AND b.ValidFrom < w.end_ts
   AND b.ValidTo >= w.end_ts
  WHERE b.RegulationID IN (1, 2, 9, 11)
    AND b.AccountTypeID NOT IN (7, 9)
    AND h.LabelID NOT IN (26, 30)
)
SELECT
  ca.CID, ca.GCID, ca.PlayerLevelID, ca.PlayerStatusID, ca.CountryID,
  ca.LabelID, ca.FirstName, ca.LastName, ca.BirthDate,
  ca.RegulationID, ca.AccountTypeID, ca.Lei, ca.CountryIDByIP,
  ca.curFirstName, ca.curLastName, ca.curBirthDate,
  ca.CitizenshipCountryID,
  (SELECT report_date FROM run_window) AS ReportDate
FROM customer_asof_ranked ca
WHERE ca.rn = 1
"""

spark.sql(customer_sql)
cnt = spark.sql("SELECT COUNT(*) AS c FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_customer").collect()[0].c
print(f"✓ mifid2_ext_customer created: {cnt:,} rows")

In [0]:
# MIFID2_ext_RegChange_Customer: Regulation-change customer snapshot
# SSIS parity: non-MiFID current regulations with migration-population constraints

regchange_sql = f"""
CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_regchange_customer
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_ext_regchange_customer'
AS
WITH run_window AS (
  SELECT
    CAST('{report_date}' AS DATE) AS report_date,
    CAST('{report_date}' AS TIMESTAMP) AS start_ts,
    CAST(date_add(CAST('{report_date}' AS DATE), 1) AS TIMESTAMP) AS end_ts
),
reg_population AS (
  SELECT p.CID, p.PrevRegulationID, p.RegulationID AS NewRegulationID
  FROM main.regtech_ops_stg.bi_output_regtechops_reg_migrationinout_population p
  JOIN run_window w ON p.RunDate = w.report_date
  WHERE p.PrevRegulationID IN (1, 2, 9, 11)
),
-- CID scope: positions active in report-day window that overlap with reg-migration population
regchange_cids AS (
  SELECT DISTINCT p.CID
  FROM main.bi_db.bronze_etoro_trade_positionforexternaluse p
  JOIN run_window w ON p.Occurred >= w.start_ts AND p.Occurred < w.end_ts
  JOIN reg_population reg ON p.CID = reg.CID
  UNION
  SELECT DISTINCT p.CID
  FROM main.trading.bronze_etoro_history_position_datafactory p
  JOIN run_window w
    ON ((p.CloseOccurred >= w.start_ts AND p.CloseOccurred < w.end_ts)
        OR (p.OpenOccurred >= w.start_ts AND p.OpenOccurred < w.end_ts AND p.CloseOccurred >= w.end_ts))
   AND p.OpenOccurred >= CAST('2015-04-26' AS TIMESTAMP)
  JOIN reg_population reg ON p.CID = reg.CID
),
customer_asof_ranked AS (
  SELECT
    c.CID, c.GCID, c.PlayerLevelID, c.PlayerStatusID, c.CountryID,
    h.LabelID, h.FirstName, h.LastName, h.BirthDate,
    b.RegulationID, b.AccountTypeID, b.Lei, c.CountryIDByIP,
    h.FirstName AS curFirstName, h.LastName AS curLastName, h.BirthDate AS curBirthDate,
    c.CitizenshipCountryID,
    ROW_NUMBER() OVER (PARTITION BY c.CID ORDER BY b.ValidFrom DESC, h.ValidFrom DESC) AS rn
  FROM {CUSTOMER_TABLE} c
  JOIN regchange_cids rc ON c.CID = rc.CID
  JOIN run_window w ON 1 = 1
  JOIN {HISTORY_CUSTOMER_TABLE} h
    ON c.CID = h.CID AND h.ValidFrom < w.end_ts AND h.ValidTo >= w.end_ts
  JOIN main.general.bronze_etoro_history_backofficecustomer b
    ON c.CID = b.CID AND b.ValidFrom < w.end_ts AND b.ValidTo >= w.end_ts
  WHERE b.RegulationID NOT IN (1, 2, 9, 11)
    AND b.AccountTypeID NOT IN (7, 9)
    AND h.LabelID NOT IN (26, 30)
)
SELECT
  ca.CID, ca.GCID, ca.PlayerLevelID, ca.PlayerStatusID, ca.CountryID,
  ca.LabelID, ca.FirstName, ca.LastName, ca.BirthDate,
  ca.RegulationID, ca.AccountTypeID, ca.Lei, ca.CountryIDByIP,
  ca.curFirstName, ca.curLastName, ca.curBirthDate,
  ca.CitizenshipCountryID,
  (SELECT report_date FROM run_window) AS ReportDate
FROM customer_asof_ranked ca
WHERE ca.rn = 1
"""

spark.sql(regchange_sql)
cnt = spark.sql("SELECT COUNT(*) AS c FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_regchange_customer").collect()[0].c
print(f"✓ mifid2_ext_regchange_customer created: {cnt:,} rows")

In [0]:
# MIFID2_ext_Position: Position staging scoped to customer CIDs
# SSIS parity: Trade.PositionForExternalUse + History.PositionForExternalUse

position_sql = f"""
CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_ext_position'
AS
WITH run_window AS (
  SELECT
    CAST('{report_date}' AS DATE) AS report_date,
    CAST('{report_date}' AS TIMESTAMP) AS start_ts,
    CAST(date_add(CAST('{report_date}' AS DATE), 1) AS TIMESTAMP) AS end_ts
),
customer_scope AS (
  SELECT DISTINCT CID FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_customer
),
-- Trade positions (current open) - map Occurred→OpenOccurred, NULL for close-only fields
trade_positions AS (
  SELECT
    p.PositionID, p.ParentPositionID, p.CID,
    p.Occurred AS OpenOccurred,
    CAST(NULL AS TIMESTAMP) AS CloseOccurred,
    p.InitForexRate,
    CAST(NULL AS DECIMAL(18,8)) AS EndForexRate,
    p.AmountInUnitsDecimal, p.InstrumentID,
    p.IsBuy, p.Leverage, p.LastOpConversionRate, p.MirrorID,
    p.InitExecutionID,
    CAST(NULL AS BIGINT) AS EndExecutionID,
    p.HedgeServerID, CAST(p.IsSettled AS INT) AS IsSettled,
    p.InitForexPriceRateID,
    CAST(NULL AS BIGINT) AS EndForexPriceRateID,
    p.LastOpPriceRate,
    p.PositionID AS OriginalPositionID,
    p.InitialUnits,
    w.report_date AS ReportDate
  FROM main.bi_db.bronze_etoro_trade_positionforexternaluse p
  JOIN run_window w ON p.Occurred >= w.start_ts AND p.Occurred < w.end_ts
  JOIN customer_scope c ON p.CID = c.CID
),
-- History positions (closed)
history_positions AS (
  SELECT
    p.PositionID, p.ParentPositionID, p.CID, p.OpenOccurred, p.CloseOccurred,
    p.InitForexRate, p.EndForexRate, p.AmountInUnitsDecimal, p.InstrumentID,
    p.IsBuy, p.Leverage, p.LastOpConversionRate, p.MirrorID,
    p.InitExecutionID, p.EndExecutionID, p.HedgeServerID, p.IsSettled,
    p.InitForexPriceRateID, p.EndForexPriceRateID, p.LastOpPriceRate,
    COALESCE(p.OriginalPositionID, p.PositionID) AS OriginalPositionID,
    p.InitialUnits,
    w.report_date AS ReportDate
  FROM main.trading.bronze_etoro_history_position_datafactory p
  JOIN run_window w
    ON ((p.CloseOccurred >= w.start_ts AND p.CloseOccurred < w.end_ts)
        OR (p.OpenOccurred >= w.start_ts AND p.OpenOccurred < w.end_ts AND COALESCE(p.CloseOccurred, w.end_ts) >= w.end_ts))
   AND p.OpenOccurred >= CAST('2015-04-26' AS TIMESTAMP)
  JOIN customer_scope c ON p.CID = c.CID
)
SELECT * FROM trade_positions
UNION ALL
SELECT * FROM history_positions
"""

spark.sql(position_sql)
cnt = spark.sql("SELECT COUNT(*) AS c FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position").collect()[0].c
print(f"✓ mifid2_ext_position created: {cnt:,} rows")

In [0]:
# MIFID2_ext_RegChange_Position: Positions for regulation-change customers
# SSIS parity: MIFID2.dtsx - position union scoped by RegChange_Customer CIDs
# Uses reg_migrationinout_population for RegValidFrom/RegValidTo interval filtering
# Position interval: OpenOccurred < COALESCE(RegValidTo,'9999-12-31') AND COALESCE(CloseOccurred,'9999-12-31') >= RegValidFrom

regchange_position_sql = f"""
CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_regchange_position
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_ext_regchange_position'
AS
WITH run_window AS (
  SELECT
    CAST('{report_date}' AS DATE) AS report_date,
    CAST('{report_date}' AS TIMESTAMP) AS start_ts,
    CAST(date_add(CAST('{report_date}' AS DATE), 1) AS TIMESTAMP) AS end_ts
),
-- RegChange CID scope from regchange_customer table
regchange_cid_scope AS (
  SELECT DISTINCT CID
  FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_regchange_customer
),
-- Migration population for regulation interval constraints
-- Uses Migration_Occurred as the regulation change boundary
reg_population AS (
  SELECT p.CID, p.PrevRegulationID, p.Migration_Occurred
  FROM main.regtech_ops_stg.bi_output_regtechops_reg_migrationinout_population p
  WHERE p.PrevRegulationID IN (1, 2, 9, 11)
),
-- Position union: trade (open) + history (closed)
position_union AS (
  SELECT
    p.PositionID, p.ParentPositionID, p.CID,
    p.Occurred AS OpenOccurred,
    CAST(NULL AS TIMESTAMP) AS CloseOccurred,
    p.InitForexRate,
    CAST(NULL AS DECIMAL(18,8)) AS EndForexRate,
    p.AmountInUnitsDecimal, p.InstrumentID,
    p.IsBuy, p.Leverage, p.LastOpConversionRate, p.MirrorID,
    p.InitExecutionID,
    CAST(NULL AS BIGINT) AS EndExecutionID,
    p.HedgeServerID, CAST(p.IsSettled AS INT) AS IsSettled,
    p.InitForexPriceRateID,
    CAST(NULL AS BIGINT) AS EndForexPriceRateID,
    p.LastOpPriceRate,
    p.PositionID AS OriginalPositionID,
    p.InitialUnits
  FROM main.bi_db.bronze_etoro_trade_positionforexternaluse p
  JOIN run_window w ON p.Occurred >= w.start_ts AND p.Occurred < w.end_ts
  JOIN regchange_cid_scope rc ON p.CID = rc.CID
  UNION ALL
  SELECT
    p.PositionID, p.ParentPositionID, p.CID, p.OpenOccurred, p.CloseOccurred,
    p.InitForexRate, p.EndForexRate, p.AmountInUnitsDecimal, p.InstrumentID,
    p.IsBuy, p.Leverage, p.LastOpConversionRate, p.MirrorID,
    p.InitExecutionID, p.EndExecutionID, p.HedgeServerID, p.IsSettled,
    p.InitForexPriceRateID, p.EndForexPriceRateID, p.LastOpPriceRate,
    COALESCE(p.OriginalPositionID, p.PositionID) AS OriginalPositionID,
    p.InitialUnits
  FROM main.trading.bronze_etoro_history_position_datafactory p
  JOIN run_window w
    ON ((p.CloseOccurred >= w.start_ts AND p.CloseOccurred < w.end_ts)
        OR (p.OpenOccurred >= w.start_ts AND p.OpenOccurred < w.end_ts AND COALESCE(p.CloseOccurred, w.end_ts) >= w.end_ts))
   AND p.OpenOccurred >= CAST('2015-04-26' AS TIMESTAMP)
  JOIN regchange_cid_scope rc ON p.CID = rc.CID
)
-- Final: filter positions active under previous regulation (opened before migration)
SELECT
  pu.*,
  rp.PrevRegulationID,
  rp.Migration_Occurred,
  (SELECT report_date FROM run_window) AS ReportDate
FROM position_union pu
JOIN reg_population rp
  ON pu.CID = rp.CID
 AND pu.OpenOccurred < rp.Migration_Occurred
"""

spark.sql(regchange_position_sql)
cnt = spark.sql("SELECT COUNT(*) AS c FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_regchange_position").collect()[0].c
print(f"\u2713 mifid2_ext_regchange_position created: {cnt:,} rows")

In [0]:
%sql
-- Validation: Module 8B staging tables
SELECT 'mifid2_ext_customer' AS table_name, COUNT(*) AS row_count, COUNT(DISTINCT CID) AS distinct_cids FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_customer
UNION ALL
SELECT 'mifid2_ext_regchange_customer', COUNT(*), COUNT(DISTINCT CID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_regchange_customer
UNION ALL
SELECT 'mifid2_ext_position', COUNT(*), COUNT(DISTINCT CID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_position
UNION ALL
SELECT 'mifid2_ext_regchange_position', COUNT(*), COUNT(DISTINCT CID) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_regchange_position